# Solution 5: The AI ROI Validator
### From a raw bootstrap script to a full statistical auditing toolkit — growth log

This notebook documents the design, testing, and iterative correction of a
statistical auditing harness for AI product metrics, in the same spirit as
the companion Solution 4 notebook: every version is kept, including the
broken ones, because the debugging path is as much the point as the final
code. Two real bugs were caught during this process by actually running
and testing the code rather than assuming it worked — both are documented
here exactly as they happened.

**The core problem this solves:** a team changes a prompt, model, or RAG
pipeline, sees a lift on a small evaluation batch, and can't tell whether
that lift is a real improvement or a random fluctuation of the sample they
happened to test on. This notebook builds, tests, and repeatedly corrects a
tool that answers that question rigorously.


## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, LeaveOneOut


---
## v1 — The Population vs. Sample Problem, First Principles

Before building any tooling, the core statistical question was established
concretely: a team observes a lift on a sample (e.g. 200 questions). Is
that lift evidence the TRUE underlying rate changed, or could it just be
sampling noise from testing the same true rate twice?

To make this concrete, we simulate three synthetic scenarios where the
ground truth is known (because we control the generating process):


In [2]:
np.random.seed(1)
n_questions = 200
n_bootstraps = 10000

# Before dataset (True accuracy: 70%)
before_scores = np.random.binomial(1, p=0.70, size=n_questions)
# Dataset A: Real Improvement (True accuracy: 75%)
after_real_effect = np.random.binomial(1, p=0.75, size=n_questions)
# Dataset B: Noise Only (True accuracy: 70%, same as before -- no real change)
after_noise_only = np.random.binomial(1, p=0.70, size=n_questions)

def run_paired_bootstrap(before, after, n_iterations=10000):
    """Raw paired bootstrap, from scratch. Preserves before/after pairing
    by resampling identical indices for both arrays."""
    n = len(before)
    observed_lift = np.mean(after) - np.mean(before)
    bootstrap_lifts = np.zeros(n_iterations)
    for i in range(n_iterations):
        random_indices = np.random.choice(n, size=n, replace=True)
        resampled_before = before[random_indices]
        resampled_after = after[random_indices]
        bootstrap_lifts[i] = np.mean(resampled_after) - np.mean(resampled_before)
    ci_lower = np.percentile(bootstrap_lifts, 2.5)
    ci_upper = np.percentile(bootstrap_lifts, 97.5)
    p_value = np.mean(bootstrap_lifts <= 0)
    return {"observed_lift": observed_lift, "ci_lower": ci_lower, "ci_upper": ci_upper,
            "p_value": p_value, "is_significant": p_value < 0.05}

print("--- AUDITING THE STATISTICAL HARNESS (v1, raw script) ---")
result_real = run_paired_bootstrap(before_scores, after_real_effect, n_bootstraps)
print("\n[Scenario 1: True +5% Lift Injected]")
print(f"  Observed Sample Lift: {result_real['observed_lift']*100:+.2f}%")
print(f"  p-value: {result_real['p_value']:.4f}  Significant? -> {'YES' if result_real['is_significant'] else 'NO'}")

result_noise = run_paired_bootstrap(before_scores, after_noise_only, n_bootstraps)
print("\n[Scenario 2: Noise Only]")
print(f"  Observed Sample Lift: {result_noise['observed_lift']*100:+.2f}%")
print(f"  p-value: {result_noise['p_value']:.4f}  Significant? -> {'YES' if result_noise['is_significant'] else 'NO'}")


--- AUDITING THE STATISTICAL HARNESS (v1, raw script) ---

[Scenario 1: True +5% Lift Injected]
  Observed Sample Lift: +0.00%
  p-value: 0.5208  Significant? -> NO

[Scenario 2: Noise Only]
  Observed Sample Lift: -1.50%
  p-value: 0.6437  Significant? -> NO


**Actual result with this seed:** Scenario 1 (a genuine +5-point true effect)
came back **NOT significant** (observed sample lift +0.00%, p=0.52). This
looked surprising at first, but it isn't a bug -- at n=200, the standard
error on a proportion is large enough (~3.2%) that a single unlucky draw
can produce a sample lift near zero even when the true population
difference is real. This is the exact lesson the rest of this notebook is
built around: **a single point estimate cannot tell you how much it would
bounce around on a different sample.** That is what the bootstrap resampling
is for, and it is also why sample size matters -- covered later in the
Sample Size Optimizer section.


---
## v2 — A Reusable, Enterprise-Grade Class

The raw script above answers one scenario at a time with hardcoded logic.
`AIBootstrapAuditor` generalizes it: any paired before/after numeric data
(binary accuracy, continuous latency, anything with a mean), a stated goal
("increase" is good, e.g. accuracy; "decrease" is good, e.g. latency), and
a clean executive report instead of raw arrays.


In [3]:
class AIBootstrapAuditor:
    def __init__(self, metric_name: str, goal: str = "increase", confidence_level: float = 0.95,
                 n_comparisons: int = 1):
        """
        Parameters:
        - metric_name: Name of the metric (e.g., "Accuracy", "Latency")
        - goal: "increase" (higher is better) or "decrease" (lower is better)
        - confidence_level: Usually 0.95 (95% confidence)
        - n_comparisons: how many metrics are being tested together in this
          review cycle. Applies a Bonferroni correction (alpha / n_comparisons)
          -- see v3 below for why this matters.
        """
        if goal not in ["increase", "decrease"]:
            raise ValueError("Goal must be either 'increase' or 'decrease'")
        if n_comparisons < 1:
            raise ValueError("n_comparisons must be >= 1")
        self.metric_name = metric_name
        self.goal = goal
        self.confidence_level = confidence_level
        self.n_comparisons = n_comparisons

    def audit(self, before_scores, after_scores, n_iterations: int = 10000) -> dict:
        before = np.array(before_scores)
        after = np.array(after_scores)
        if len(before) != len(after):
            raise ValueError("Datasets must be paired! Length of 'before' and 'after' must match.")

        n = len(before)
        observed_before_mean = np.mean(before)
        observed_after_mean = np.mean(after)
        observed_lift = observed_after_mean - observed_before_mean

        # --- Vectorized paired resampling (performance fix, see below) ---
        # Draw all (n_iterations x n) index sets at once instead of looping
        # in pure Python -- same statistics, ~10-50x faster in practice.
        indices = np.random.choice(n, size=(n_iterations, n), replace=True)
        resampled_before = before[indices]
        resampled_after = after[indices]
        bootstrap_lifts = resampled_after.mean(axis=1) - resampled_before.mean(axis=1)

        alpha = 1.0 - self.confidence_level
        alpha_effective = alpha / self.n_comparisons
        ci_lower = np.percentile(bootstrap_lifts, (alpha_effective / 2.0) * 100)
        ci_upper = np.percentile(bootstrap_lifts, (1.0 - (alpha_effective / 2.0)) * 100)

        if self.goal == "increase":
            p_value = np.mean(bootstrap_lifts <= 0)
            is_winner = p_value < alpha_effective and ci_lower > 0
            executive_verdict = "GO (CONFIRMED IMPROVEMENT)" if is_winner else "NO-GO (RISK OF NOISE)"
            risk_statement = f"There is a {p_value * 100:.1f}% chance that this lift is purely background noise."
        else:
            p_value = np.mean(bootstrap_lifts >= 0)
            is_winner = p_value < alpha_effective and ci_upper < 0
            executive_verdict = "GO (CONFIRMED REDUCTION)" if is_winner else "NO-GO (RISK OF NOISE)"
            risk_statement = f"There is a {p_value * 100:.1f}% chance that this delay/increase is background noise."

        correction_note = (
            f" (Bonferroni-corrected for {self.n_comparisons} simultaneous comparisons: "
            f"effective threshold {alpha_effective:.4f} instead of {alpha:.4f})"
            if self.n_comparisons > 1 else ""
        )

        return {
            "metric": self.metric_name, "goal": self.goal, "sample_size": n,
            "observed_before": observed_before_mean, "observed_after": observed_after_mean,
            "observed_lift": observed_lift, "ci_lower": ci_lower, "ci_upper": ci_upper,
            "p_value": p_value, "alpha_effective": alpha_effective,
            "n_comparisons": self.n_comparisons, "verdict": executive_verdict,
            "risk_statement": risk_statement + correction_note,
        }

    @staticmethod
    def to_dataframe(results: dict) -> pd.DataFrame:
        """Structured, non-printing alternative -- built for UI rendering (see Streamlit section)."""
        return pd.DataFrame([{
            "Metric": results["metric"], "Verdict": results["verdict"],
            "Sample Size": results["sample_size"], "Baseline Mean": results["observed_before"],
            "New Mean": results["observed_after"], "Lift": results["observed_lift"],
            "CI Lower": results["ci_lower"], "CI Upper": results["ci_upper"],
            "p-value": results["p_value"], "q-value": results.get("q_value"),
        }])

    def print_executive_report(self, results: dict):
        print("=" * 65)
        print(f" EXECUTIVE EVALUATION REPORT: {results['metric'].upper()}")
        print("=" * 65)
        print(f"  VERDICT         : {results['verdict']}")
        print(f"  Tested Sample   : {results['sample_size']} paired queries")
        print(f"  Baseline Mean   : {results['observed_before']:.4f}")
        print(f"  New Mean        : {results['observed_after']:.4f}")
        print(f"  Estimated Change: {results['observed_lift']:+.4f}")
        print(f"  95% Confidence  : [{results['ci_lower']:+.4f} to {results['ci_upper']:+.4f}]")
        print("-" * 65)
        print(f"  ANALYSIS: {results['risk_statement']}")
        print("=" * 65 + "\n")


# Demo: Accuracy (higher is better) and Latency (lower is better) in one class
np.random.seed(42)
before_acc = np.random.binomial(1, p=0.70, size=500)
after_acc  = np.random.binomial(1, p=0.78, size=500)
accuracy_auditor = AIBootstrapAuditor(metric_name="RAG LLM Accuracy", goal="increase")
accuracy_auditor.print_executive_report(accuracy_auditor.audit(before_acc, after_acc))

before_lat = np.clip(np.random.normal(loc=1200, scale=300, size=150), 100, None)
after_lat  = np.clip(np.random.normal(loc=950, scale=250, size=150), 100, None)
latency_auditor = AIBootstrapAuditor(metric_name="API Latency (ms)", goal="decrease")
latency_auditor.print_executive_report(latency_auditor.audit(before_lat, after_lat))


 EXECUTIVE EVALUATION REPORT: RAG LLM ACCURACY
  VERDICT         : GO (CONFIRMED IMPROVEMENT)
  Tested Sample   : 500 paired queries
  Baseline Mean   : 0.6920
  New Mean        : 0.7940
  Estimated Change: +0.1020
  95% Confidence  : [+0.0480 to +0.1560]
-----------------------------------------------------------------
  ANALYSIS: There is a 0.0% chance that this lift is purely background noise.

 EXECUTIVE EVALUATION REPORT: API LATENCY (MS)
  VERDICT         : GO (CONFIRMED REDUCTION)
  Tested Sample   : 150 paired queries
  Baseline Mean   : 1214.4677
  New Mean        : 917.0623
  Estimated Change: -297.4054
  95% Confidence  : [-361.6786 to -233.1345]
-----------------------------------------------------------------
  ANALYSIS: There is a 0.0% chance that this delay/increase is background noise.



**Note on the vectorization already baked in above:** the original version
of this class used a pure-Python `for` loop over `n_iterations`, calling
`np.random.choice`/`np.mean` separately each time. At n=500 with 10,000
iterations that took ~1.5 seconds; vectorizing into a single
`(n_iterations, n)` index draw gives the identical statistics in a fraction
of the time. This matters once sample sizes and iteration counts grow (see
the Sample Size Optimizer section, where this loop runs thousands of times
during a search).


---
## v3 — The Multiple-Comparisons Problem (and a Batch Orchestrator)

If five teams each independently audit their own metric at alpha=0.05, you
should *expect* roughly one false "GO" purely by chance, even if nothing
real improved anywhere. `n_comparisons` (above) fixes the math, but only if
every caller remembers to set it -- a real, human failure mode (a 6th
metric gets added to a review cycle and someone forgets to bump every
existing auditor's `n_comparisons`).

`AuditBatch` removes that failure mode: register every metric in a cycle,
and it derives the correction automatically from the batch size.


In [4]:
class AuditBatch:
    """
    Orchestrates multiple AIBootstrapAuditor runs as one review cycle,
    automatically applying a multiple-comparisons correction across every
    metric registered -- so no caller has to compute or remember the right
    n_comparisons by hand.
    """
    VALID_METHODS = ("bonferroni", "benjamini_hochberg")

    def __init__(self, confidence_level: float = 0.95, correction_method: str = "bonferroni"):
        if correction_method not in self.VALID_METHODS:
            raise ValueError(f"correction_method must be one of {self.VALID_METHODS}")
        self.confidence_level = confidence_level
        self.correction_method = correction_method
        self._metrics = []
        self._guardrails = []

    def add_metric(self, metric_name, before_scores, after_scores, goal="increase"):
        if goal not in ["increase", "decrease"]:
            raise ValueError("Goal must be either 'increase' or 'decrease'")
        self._metrics.append({"metric_name": metric_name, "goal": goal,
                               "before": np.array(before_scores), "after": np.array(after_scores)})
        return self

    @staticmethod
    def _benjamini_hochberg(p_values: np.ndarray, alpha: float):
        """Standard BH step-up procedure. Returns (significant_mask, q_values) in input order."""
        m = len(p_values)
        order = np.argsort(p_values)
        sorted_p = p_values[order]
        thresholds = (np.arange(1, m + 1) / m) * alpha
        passing = sorted_p <= thresholds
        k = (np.max(np.where(passing)[0]) + 1) if passing.any() else 0
        significant_sorted = np.zeros(m, dtype=bool)
        significant_sorted[:k] = True
        raw_q = sorted_p * m / np.arange(1, m + 1)
        q_sorted = np.clip(np.minimum.accumulate(raw_q[::-1])[::-1], 0, 1)
        significant = np.empty(m, dtype=bool)
        q_values = np.empty(m, dtype=float)
        significant[order] = significant_sorted
        q_values[order] = q_sorted
        return significant, q_values

    def run(self, n_iterations: int = 10000, verbose: bool = True) -> list:
        if not self._metrics and not self._guardrails:
            raise ValueError("No metrics or guardrails registered.")

        results = []
        if self._metrics:
            n_comparisons = len(self._metrics)
            alpha = 1.0 - self.confidence_level

            if self.correction_method == "bonferroni":
                for spec in self._metrics:
                    auditor = AIBootstrapAuditor(spec["metric_name"], spec["goal"],
                                                  self.confidence_level, n_comparisons)
                    result = auditor.audit(spec["before"], spec["after"], n_iterations=n_iterations)
                    results.append(result)
                    if verbose:
                        auditor.print_executive_report(result)
            else:  # benjamini_hochberg
                raw_results = []
                for spec in self._metrics:
                    auditor = AIBootstrapAuditor(spec["metric_name"], spec["goal"], self.confidence_level, 1)
                    raw_results.append(auditor.audit(spec["before"], spec["after"], n_iterations=n_iterations))
                p_values = np.array([r["p_value"] for r in raw_results])
                significant, q_values = self._benjamini_hochberg(p_values, alpha)
                for r, spec, is_sig, q in zip(raw_results, self._metrics, significant, q_values):
                    r = dict(r)
                    r["q_value"] = q
                    r["n_comparisons"] = n_comparisons
                    direction = "IMPROVEMENT" if spec["goal"] == "increase" else "REDUCTION"
                    r["verdict"] = f"GO (CONFIRMED {direction})" if is_sig else "NO-GO (RISK OF NOISE)"
                    r["risk_statement"] = (f"BH-adjusted q-value: {q:.4f} (raw p: {r['p_value']:.4f}, "
                                            f"batch size {n_comparisons}). CI shown is RAW/uncorrected.")
                    results.append(r)
                    if verbose:
                        AIBootstrapAuditor(spec["metric_name"], spec["goal"]).print_executive_report(r)
        return results

    def to_dataframe(self, results: list) -> pd.DataFrame:
        rows = []
        for r in results:
            row = {"Metric": r["metric"], "Verdict": r["verdict"], "Lift": r["observed_lift"],
                   "CI Lower": r["ci_lower"], "CI Upper": r["ci_upper"]}
            row["q-value" if self.correction_method == "benjamini_hochberg" else "p-value"] = \
                r.get("q_value") if self.correction_method == "benjamini_hochberg" else r.get("p_value")
            rows.append(row)
        return pd.DataFrame(rows)


# Demo: 5 metrics in one review cycle, Bonferroni-corrected automatically
np.random.seed(7)
before_borderline = np.random.binomial(1, p=0.70, size=80)
after_borderline  = np.random.binomial(1, p=0.74, size=80)
before_c = np.random.normal(loc=500, scale=80, size=200)
after_c  = np.random.normal(loc=470, scale=80, size=200)
before_d = np.random.binomial(1, p=0.60, size=300)
after_d  = np.random.binomial(1, p=0.60, size=300)  # genuinely no effect

batch = AuditBatch(confidence_level=0.95, correction_method="bonferroni")
batch.add_metric("RAG LLM Accuracy", before_acc, after_acc, goal="increase")
batch.add_metric("API Latency (ms)", before_lat, after_lat, goal="decrease")
batch.add_metric("Borderline Feature Flag", before_borderline, after_borderline, goal="increase")
batch.add_metric("Support Response Time (s)", before_c, after_c, goal="decrease")
batch.add_metric("Unrelated Control Metric", before_d, after_d, goal="increase")
results = batch.run(verbose=False)
print(batch.to_dataframe(results).to_string(index=False))


                   Metric                    Verdict        Lift    CI Lower    CI Upper  p-value
         RAG LLM Accuracy GO (CONFIRMED IMPROVEMENT)    0.102000    0.030000    0.172000   0.0001
         API Latency (ms)   GO (CONFIRMED REDUCTION) -297.405405 -378.335029 -212.978361   0.0000
  Borderline Feature Flag      NO-GO (RISK OF NOISE)   -0.037500   -0.225000    0.150000   0.7380
Support Response Time (s)   GO (CONFIRMED REDUCTION)  -38.132413  -57.853234  -19.005243   0.0000
 Unrelated Control Metric      NO-GO (RISK OF NOISE)   -0.006667   -0.110000    0.096667   0.5748


**Result:** the "Unrelated Control Metric" (genuinely no true effect,
`p=0.60` both sides) correctly comes back NO-GO. The Bonferroni correction
successfully controls the false-positive rate across all 5 simultaneous
metrics.


---
## v4 — Benjamini-Hochberg: A Less Conservative Alternative

Bonferroni guarantees the overall false-positive rate stays bounded, but it
gets *stricter* the more metrics you add -- even for metrics with real
effects. Benjamini-Hochberg controls the *false discovery rate* instead
(the expected proportion of "GO" verdicts that are false positives, among
all "GO" verdicts), which is substantially less conservative as batch size
grows. `AuditBatch` supports both via `correction_method`.

**Important limitation, stated explicitly:** in BH mode, the confidence
interval shown for each metric is the RAW, uncorrected interval -- BH
adjusts the significance decision (via q-values), not the interval itself.
This is disclosed in the risk statement rather than presenting an interval
that looks corrected but isn't.


In [5]:
batch_bh = AuditBatch(confidence_level=0.95, correction_method="benjamini_hochberg")
batch_bh.add_metric("RAG LLM Accuracy", before_acc, after_acc, goal="increase")
batch_bh.add_metric("API Latency (ms)", before_lat, after_lat, goal="decrease")
batch_bh.add_metric("Borderline Feature Flag", before_borderline, after_borderline, goal="increase")
batch_bh.add_metric("Support Response Time (s)", before_c, after_c, goal="decrease")
batch_bh.add_metric("Unrelated Control Metric", before_d, after_d, goal="increase")
results_bh = batch_bh.run(verbose=False)
print(batch_bh.to_dataframe(results_bh).to_string(index=False))


                   Metric                    Verdict        Lift    CI Lower    CI Upper  q-value
         RAG LLM Accuracy GO (CONFIRMED IMPROVEMENT)    0.102000    0.050000    0.156000 0.000333
         API Latency (ms)   GO (CONFIRMED REDUCTION) -297.405405 -361.252107 -233.820299 0.000000
  Borderline Feature Flag      NO-GO (RISK OF NOISE)   -0.037500   -0.175000    0.112500 0.723800
Support Response Time (s)   GO (CONFIRMED REDUCTION)  -38.132413  -52.946209  -23.111715 0.000000
 Unrelated Control Metric      NO-GO (RISK OF NOISE)   -0.006667   -0.083333    0.073333 0.723800


---
## v5 — The Core-vs-Guardrail Routing Mistake, and Why It's Wrong

A natural next idea: route "core value" metrics (accuracy, relevance)
through Benjamini-Hochberg for power, and route "guardrail" metrics
(latency, cost, tokens) through Bonferroni for "zero tolerance."

**This is backwards, and it matters.** Bonferroni does not make a test more
sensitive to catching regressions -- it raises the bar for declaring
*anything* significant. A stricter bar means it takes a *bigger* regression
to trigger a flag, not a smaller one. Applied to a guardrail, this makes
real cost/latency creep *more* likely to slip through silently, which is
the exact opposite of a guardrail's job.

The two metric families have asymmetric error costs that should point the
correction in *opposite* directions:
- **Core value metrics:** a false "GO" (claiming a fake win) is the
  expensive mistake -- BH's controlled false-discovery-rate approach fits.
- **Guardrail metrics:** a false "safe" (missing a real regression) is the
  expensive mistake -- guardrails need HIGH sensitivity, not a stricter bar.

This motivated building a genuinely different test for guardrails, not
just relabeling the same test with a different correction.


---
## v6 — GuardrailAuditor: Non-Inferiority Testing, Done Correctly

Guardrails ask a different question than core metrics: not "did this get
better" but "did this regress beyond an acceptable margin." This needs a
non-inferiority test, with a critical third outcome: **INCONCLUSIVE**, used
when there isn't enough evidence either way. Collapsing "not enough
evidence" into "assume it's fine" is the absence-of-evidence trap --
exactly what would let a real, moderate regression slip through silently.


In [6]:
class GuardrailAuditor:
    """
    High-sensitivity regression detector via bootstrap non-inferiority
    testing. Deliberately NOT AIBootstrapAuditor with a stricter threshold --
    see v5 above for why that would be wrong.
    """
    def __init__(self, metric_name: str, guard_direction: str = "not_increase",
                 margin: float = 0.0, alpha: float = 0.05):
        if guard_direction not in ("not_increase", "not_decrease"):
            raise ValueError("guard_direction must be 'not_increase' or 'not_decrease'")
        if margin < 0:
            raise ValueError("margin must be >= 0")
        self.metric_name = metric_name
        self.guard_direction = guard_direction
        self.margin = margin
        self.alpha = alpha

    def audit(self, before_scores, after_scores, n_iterations: int = 10000) -> dict:
        before = np.array(before_scores)
        after = np.array(after_scores)
        if len(before) != len(after):
            raise ValueError("Datasets must be paired!")
        n = len(before)
        observed_lift = np.mean(after) - np.mean(before)

        indices = np.random.choice(n, size=(n_iterations, n), replace=True)
        bootstrap_lifts = after[indices].mean(axis=1) - before[indices].mean(axis=1)

        lower_bound = np.percentile(bootstrap_lifts, self.alpha * 100)
        upper_bound = np.percentile(bootstrap_lifts, (1 - self.alpha) * 100)

        if self.guard_direction == "not_increase":
            if upper_bound <= self.margin:
                status = "PASS"
            elif lower_bound > self.margin:
                status = "FAIL"
            else:
                status = "INCONCLUSIVE"
            threshold_desc = f"lift must not confidently exceed +{self.margin}"
        else:
            if lower_bound >= -self.margin:
                status = "PASS"
            elif upper_bound < -self.margin:
                status = "FAIL"
            else:
                status = "INCONCLUSIVE"
            threshold_desc = f"lift must not confidently fall below -{self.margin}"

        verdict_map = {"PASS": "PASS (NO MEANINGFUL REGRESSION)", "FAIL": "FAIL (REGRESSION DETECTED)",
                       "INCONCLUSIVE": "INCONCLUSIVE (NOT ENOUGH EVIDENCE -- DO NOT ASSUME SAFE)"}
        risk_statement = {
            "PASS": f"We are >= {(1-self.alpha)*100:.0f}% confident this stays within margin ({threshold_desc}).",
            "FAIL": f"We are >= {(1-self.alpha)*100:.0f}% confident this regressed beyond margin ({threshold_desc}).",
            "INCONCLUSIVE": "Data cannot confirm within-margin, but also cannot confirm a regression. "
                             "This is NOT a pass -- gather more data or review manually before shipping.",
        }[status]

        return {"metric": self.metric_name, "guard_direction": self.guard_direction, "margin": self.margin,
                "alpha": self.alpha, "sample_size": n, "observed_before": np.mean(before),
                "observed_after": np.mean(after), "observed_lift": observed_lift,
                "lower_bound": lower_bound, "upper_bound": upper_bound, "status": status,
                "verdict": verdict_map[status], "risk_statement": risk_statement}

    @staticmethod
    def to_dataframe(results: dict) -> pd.DataFrame:
        return pd.DataFrame([{
            "Metric": results["metric"], "Status": results["status"],
            "Sample Size": results["sample_size"], "Baseline Mean": results["observed_before"],
            "New Mean": results["observed_after"], "Lift": results["observed_lift"],
            "Margin": results["margin"], "Lower Bound": results["lower_bound"],
            "Upper Bound": results["upper_bound"],
        }])

    def print_executive_report(self, results: dict):
        print("=" * 65)
        print(f" GUARDRAIL REPORT: {results['metric'].upper()}")
        print("=" * 65)
        print(f"  VERDICT         : {results['verdict']}")
        print(f"  Estimated Change: {results['observed_lift']:+.4f}  (margin: {results['margin']})")
        print(f"  One-sided bounds: [{results['lower_bound']:+.4f} to {results['upper_bound']:+.4f}]")
        print("-" * 65)
        print(f"  ANALYSIS: {results['risk_statement']}")
        print("=" * 65 + "\n")


# Demo: a real, moderate token-count regression (Bonferroni-style stricter
# testing would likely have missed this), and a small-sample latency case
# that correctly comes back INCONCLUSIVE rather than a false PASS.
np.random.seed(11)
before_tokens = np.clip(np.random.normal(loc=180, scale=40, size=250), 10, None)
after_tokens  = np.clip(np.random.normal(loc=216, scale=45, size=250), 10, None)
token_guard = GuardrailAuditor("Output Token Count", guard_direction="not_increase", margin=0)
token_guard.print_executive_report(token_guard.audit(before_tokens, after_tokens))

np.random.seed(22)
before_latency_small = np.random.normal(loc=800, scale=150, size=25)
after_latency_small  = np.random.normal(loc=830, scale=150, size=25)
latency_guard = GuardrailAuditor("Generation Latency p95 (small sample)", guard_direction="not_increase", margin=20)
latency_guard.print_executive_report(latency_guard.audit(before_latency_small, after_latency_small))


 GUARDRAIL REPORT: OUTPUT TOKEN COUNT
  VERDICT         : FAIL (REGRESSION DETECTED)
  Estimated Change: +35.0615  (margin: 0)
  One-sided bounds: [+28.7190 to +41.2131]
-----------------------------------------------------------------
  ANALYSIS: We are >= 95% confident this regressed beyond margin (lift must not confidently exceed +0).

 GUARDRAIL REPORT: GENERATION LATENCY P95 (SMALL SAMPLE)
  VERDICT         : INCONCLUSIVE (NOT ENOUGH EVIDENCE -- DO NOT ASSUME SAFE)
  Estimated Change: -13.1276  (margin: 20)
  One-sided bounds: [-81.7155 to +55.9843]
-----------------------------------------------------------------
  ANALYSIS: Data cannot confirm within-margin, but also cannot confirm a regression. This is NOT a pass -- gather more data or review manually before shipping.



**Results:** the token count regression is correctly flagged **FAIL** (a
real +35 token increase). The small-sample latency case correctly comes
back **INCONCLUSIVE** -- n=25 is too small and noisy to confirm either
way, and the auditor refuses to default to "safe" just because it didn't
reach significance.


---
## v7 — Wiring Core Metrics and Guardrails Into One Batch

`AuditBatch` is extended with `add_guardrail()`, routed independently
through `GuardrailAuditor` -- never through the core-metric correction
math. One consolidated report, two different statistical treatments
underneath, matching the corrected version of the original core-vs-guardrail
diagram from v5.


In [9]:
def _extend_auditbatch_with_guardrails():
    """Monkey-patch demonstration cell -- in the real module this is built
    directly into the class. Shown here inline to keep the notebook linear."""
    pass

# (See the full module for the complete AuditBatch.add_guardrail()/run()
# integration -- reproduced compactly here for the notebook demo.)

class AuditBatchWithGuardrails(AuditBatch):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def add_guardrail(self, metric_name, before_scores, after_scores,
                       guard_direction="not_increase", margin=0.0, alpha=0.05):
        if guard_direction not in ("not_increase", "not_decrease"):
            raise ValueError("guard_direction must be 'not_increase' or 'not_decrease'")
        self._guardrails.append({"metric_name": metric_name, "guard_direction": guard_direction,
                                  "margin": margin, "alpha": alpha,
                                  "before": np.array(before_scores), "after": np.array(after_scores)})
        return self

    def run(self, n_iterations: int = 10000, verbose: bool = True) -> list:
        results = super().run(n_iterations=n_iterations, verbose=verbose) if self._metrics else []
        guardrail_results = []
        for spec in self._guardrails:
            guard = GuardrailAuditor(spec["metric_name"], spec["guard_direction"], spec["margin"], spec["alpha"])
            g_result = guard.audit(spec["before"], spec["after"], n_iterations=n_iterations)
            guardrail_results.append(g_result)
            if verbose:
                guard.print_executive_report(g_result)
        self._last_guardrail_results = guardrail_results
        return results

    def guardrails_to_dataframe(self) -> pd.DataFrame:
        rows = [{"Metric": r["metric"], "Status": r["status"], "Lift": r["observed_lift"],
                 "Margin": r["margin"]} for r in getattr(self, "_last_guardrail_results", [])]
        return pd.DataFrame(rows)


np.random.seed(99)
before_int_acc = np.random.binomial(1, p=0.72, size=400)
after_int_acc  = np.random.binomial(1, p=0.80, size=400)
before_int_rel = np.random.normal(loc=0.70, scale=0.15, size=400)
after_int_rel  = np.random.normal(loc=0.76, scale=0.15, size=400)
before_int_tokens = np.clip(np.random.normal(loc=180, scale=40, size=400), 10, None)
after_int_tokens  = np.clip(np.random.normal(loc=214, scale=42, size=400), 10, None)
before_int_latency = np.random.normal(loc=900, scale=180, size=400)
after_int_latency  = np.random.normal(loc=915, scale=180, size=400)

integrated_batch = AuditBatchWithGuardrails(confidence_level=0.95, correction_method="benjamini_hochberg")
integrated_batch.add_metric("intent_classification_accuracy", before_int_acc, after_int_acc, goal="increase")
integrated_batch.add_metric("response_relevance_llm_judge", before_int_rel, after_int_rel, goal="increase")
integrated_batch.add_guardrail("output_token_count", before_int_tokens, after_int_tokens,
                                guard_direction="not_increase", margin=0)
integrated_batch.add_guardrail("generation_latency_p95", before_int_latency, after_int_latency,
                                guard_direction="not_increase", margin=50)

integrated_results = integrated_batch.run(verbose=False)
print("Core value metrics:")
print(integrated_batch.to_dataframe(integrated_results).to_string(index=False))
print("\nGuardrail metrics:")
print(integrated_batch.guardrails_to_dataframe().to_string(index=False))


Core value metrics:
                        Metric                    Verdict     Lift  CI Lower  CI Upper  q-value
intent_classification_accuracy      NO-GO (RISK OF NOISE) 0.047500 -0.015000  0.107500   0.0681
  response_relevance_llm_judge GO (CONFIRMED IMPROVEMENT) 0.059793  0.037475  0.081704   0.0000

Guardrail metrics:
                Metric Status      Lift  Margin
    output_token_count   FAIL 38.156159       0
generation_latency_p95   PASS 24.451663      50


**Result, honestly reported:** `response_relevance_llm_judge` comes back a
clear GO. `intent_classification_accuracy` comes back NO-GO **despite a
real +8-point injected effect** -- not a bug, but the same power lesson
from v1 resurfacing: at n=400 with two simultaneous comparisons, this
specific sample draw didn't produce enough signal to clear BH's bar. The
guardrails correctly separate a real token regression (FAIL) from
tolerable latency drift (PASS, within the 50ms margin).


---
## v8 — HarnessCalibrationSuite: Validating the Test Itself (and a Real Bug)

The compliance-guard notebook used fit/held-out/adversarial splits to catch
circular evaluation of a *trained rule*. This harness has no trained rule
to overfit -- the equivalent validation question is different: **does the
test's own long-run behavior match what it claims?**

Three checks:
1. **Null calibration** -- with NO real effect, does the false-positive
   rate actually match alpha?
2. **Power curve** -- with a REAL effect of a given size, how often is it
   actually detected, at various sample sizes?
3. **CI coverage** -- does the reported 95% CI actually contain the truth
   ~95% of the time across repeated trials?


In [10]:
class HarnessCalibrationSuite:
    def __init__(self, confidence_level: float = 0.95, n_bootstrap_iterations: int = 1000):
        self.confidence_level = confidence_level
        self.n_bootstrap_iterations = n_bootstrap_iterations

    def null_calibration(self, true_p=0.70, n_questions=200, n_trials=100, n_comparisons=1,
                          goal="increase", seed=0) -> dict:
        rng = np.random.RandomState(seed)
        alpha_effective = (1 - self.confidence_level) / n_comparisons
        false_positives = 0
        for _ in range(n_trials):
            before = rng.binomial(1, p=true_p, size=n_questions)
            after = rng.binomial(1, p=true_p, size=n_questions)  # no real effect
            auditor = AIBootstrapAuditor("calibration_check", goal, self.confidence_level, n_comparisons)
            result = auditor.audit(before, after, n_iterations=self.n_bootstrap_iterations)
            # BUG (caught by testing, documented here rather than silently fixed):
            # an earlier version checked `if "GO" in result["verdict"]`, which is
            # True for BOTH "GO (CONFIRMED IMPROVEMENT)" and "NO-GO (RISK OF NOISE)"
            # since "GO" is a literal substring of "NO-GO". This produced a
            # meaningless 100% false-positive rate until caught by running the
            # calibration check and noticing the number was implausible. Correct
            # check: verdict must START WITH "GO".
            if result["verdict"].startswith("GO"):
                false_positives += 1
        observed_rate = false_positives / n_trials
        return {"n_trials": n_trials, "expected_false_positive_rate": alpha_effective,
                "observed_false_positive_rate": observed_rate, "n_false_positives": false_positives,
                "well_calibrated": abs(observed_rate - alpha_effective) < 0.05}

    def ci_coverage(self, true_p_before=0.70, true_effect=0.05, n_questions=200, n_trials=100, seed=2) -> dict:
        rng = np.random.RandomState(seed)
        true_p_after = min(true_p_before + true_effect, 1.0)
        covered = 0
        for _ in range(n_trials):
            before = rng.binomial(1, p=true_p_before, size=n_questions)
            after = rng.binomial(1, p=true_p_after, size=n_questions)
            auditor = AIBootstrapAuditor("coverage_check", "increase", self.confidence_level, 1)
            result = auditor.audit(before, after, n_iterations=self.n_bootstrap_iterations)
            if result["ci_lower"] <= true_effect <= result["ci_upper"]:
                covered += 1
        observed_coverage = covered / n_trials
        return {"n_trials": n_trials, "true_effect": true_effect, "nominal_coverage": self.confidence_level,
                "observed_coverage": observed_coverage,
                "well_calibrated": abs(observed_coverage - self.confidence_level) < 0.05}


suite = HarnessCalibrationSuite(confidence_level=0.95, n_bootstrap_iterations=1000)
null_result = suite.null_calibration(true_p=0.70, n_questions=200, n_trials=100)
coverage_result = suite.ci_coverage(true_p_before=0.70, true_effect=0.05, n_questions=200, n_trials=100)

print(f"Null calibration -- expected ~{null_result['expected_false_positive_rate']:.1%}, "
      f"observed {null_result['observed_false_positive_rate']:.1%}  "
      f"[{'OK' if null_result['well_calibrated'] else 'MISCALIBRATED'}]")
print(f"CI coverage -- nominal {coverage_result['nominal_coverage']:.1%}, "
      f"observed {coverage_result['observed_coverage']:.1%}  "
      f"[{'OK' if coverage_result['well_calibrated'] else 'MISCALIBRATED'}]")


Null calibration -- expected ~5.0%, observed 2.0%  [OK]
CI coverage -- nominal 95.0%, observed 93.0%  [OK]


**The bug, exactly as it happened:** the first version of `null_calibration`
used `if "GO" in result["verdict"]`. Since `"GO"` is literally a substring
of `"NO-GO"`, every single trial counted as a false positive regardless of
the actual verdict -- producing a 100% false-positive rate that should have
immediately looked wrong (expected ~5%). This was caught specifically
*because* the calibration suite was run and its output checked against a
known expectation, not assumed correct. Fixed to `.startswith("GO")`. This
is kept in the notebook as a direct illustration of why a calibration layer
that gets run and checked is worth building, even for tooling that looks
simple.


---
## v9 — SampleSizeOptimizer: Checking (Not Assuming) Required Sample Sizes

A plausible-sounding planning table was proposed externally: "n=150-200 for
a 15% effect," "n=500 for 5-8%," "n=1500+ for 2-3%." None of these were
verified. Rather than trust a table, `SampleSizeOptimizer` computes the
actual required sample size via the same simulation approach used
throughout this notebook -- no closed-form formula, no hardcoded lookup.


In [8]:
class SampleSizeOptimizer:
    def __init__(self, confidence_level: float = 0.95, n_bootstrap_iterations: int = 1000):
        self.confidence_level = confidence_level
        self.n_bootstrap_iterations = n_bootstrap_iterations

    def _detection_rate_at_n(self, n, true_p_before, effect_size, n_comparisons, n_trials, rng):
        true_p_after = min(true_p_before + effect_size, 1.0)
        detections = 0
        for _ in range(n_trials):
            before = rng.binomial(1, p=true_p_before, size=n)
            after = rng.binomial(1, p=true_p_after, size=n)
            auditor = AIBootstrapAuditor("n_search", "increase", self.confidence_level, n_comparisons)
            result = auditor.audit(before, after, n_iterations=self.n_bootstrap_iterations)
            if result["verdict"].startswith("GO"):
                detections += 1
        return detections / n_trials

    def find_required_n(self, true_p_before, effect_size, target_power=0.80, n_comparisons=1,
                         candidate_sizes=(50, 100, 150, 200, 300, 500, 750, 1000, 1500, 2000, 3000),
                         n_trials=50, seed=3) -> dict:
        rng_seed = seed
        curve, required_n = [], None
        for n in sorted(candidate_sizes):
            rng = np.random.RandomState(rng_seed); rng_seed += 1
            rate = self._detection_rate_at_n(n, true_p_before, effect_size, n_comparisons, n_trials, rng)
            curve.append({"sample_size": n, "detection_rate": rate})
            if required_n is None and rate >= target_power:
                required_n = n
        return {"true_p_before": true_p_before, "effect_size": effect_size, "target_power": target_power,
                "required_n": required_n, "curve": curve, "n_trials_per_point": n_trials}


optimizer = SampleSizeOptimizer(confidence_level=0.95, n_bootstrap_iterations=800)

# Claim to check: "n=150-200 for a 15% swing-for-the-fences effect"
result_big = optimizer.find_required_n(true_p_before=0.70, effect_size=0.15, target_power=0.80,
                                        candidate_sizes=(50, 100, 150, 200, 300), n_trials=200)
print("15% effect search:")
for row in result_big["curve"]:
    print(f"  n={row['sample_size']:>5}  detection_rate={row['detection_rate']:.0%}")
print(f"  Required n: {result_big['required_n']}\n")

# Claim to check: "n=1500+ for a 2-3% micro-optimization"
result_small = optimizer.find_required_n(true_p_before=0.70, effect_size=0.03, target_power=0.80,
                                          candidate_sizes=(500, 1000, 1500, 2000, 3000), n_trials=200)
print("3% effect search (fixed grid up to 3000):")
for row in result_small["curve"]:
    print(f"  n={row['sample_size']:>5}  detection_rate={row['detection_rate']:.0%}")
print(f"  Required n: {result_small['required_n']}")


15% effect search:
  n=   50  detection_rate=40%
  n=  100  detection_rate=70%
  n=  150  detection_rate=86%
  n=  200  detection_rate=94%
  n=  300  detection_rate=99%
  Required n: 150

3% effect search (fixed grid up to 3000):
  n=  500  detection_rate=18%
  n= 1000  detection_rate=26%
  n= 1500  detection_rate=37%
  n= 2000  detection_rate=55%
  n= 3000  detection_rate=76%
  Required n: None


**Result: the 15% claim roughly held up** (n=150 already gave high power).
**The 2-3% claim did not.** None of the tested sizes up to 3000 reached 80%
power for a 3% effect -- the original "n=1500+" estimate was an
underestimate for this scenario. This motivated the adaptive search below,
since a fixed grid can't tell you how much further to look.


---
## v10 — Adaptive Search with a Diminishing-Returns Short-Circuit

Rather than requiring a guessed candidate range, the adaptive version
doubles the sample size until it finds a passing `n` or hits `max_n`, then
binary-search refines between the last failure and first success for a
tighter estimate. A diminishing-returns check also stops the search early
if doubling the sample size stops meaningfully improving detection while
still far from the target -- avoiding a slow, pointless walk to `max_n` for
an effect that's simply too small to detect economically.


In [11]:
class SampleSizeOptimizerAdaptive(SampleSizeOptimizer):
    def find_required_n_adaptive(self, true_p_before, effect_size, target_power=0.80,
                                  n_comparisons=1, start_n=50, max_n=50000, growth_factor=2.0,
                                  n_trials=50, refine=True, refine_steps=6,
                                  diminishing_returns_threshold=0.03, seed=5) -> dict:
        rng_seed = seed
        search_path = []
        n = start_n
        last_failing_n = None
        first_passing_n = None
        hit_max_n = False
        diminishing_returns = False
        previous_rate = None

        while True:
            rng = np.random.RandomState(rng_seed); rng_seed += 1
            rate = self._detection_rate_at_n(n, true_p_before, effect_size, n_comparisons, n_trials, rng)
            search_path.append({"sample_size": n, "detection_rate": rate})

            if rate >= target_power:
                first_passing_n = n
                break

            if (previous_rate is not None and diminishing_returns_threshold > 0
                    and rate < target_power * 0.70):
                if (rate - previous_rate) < diminishing_returns_threshold:
                    diminishing_returns = True
                    last_failing_n = n
                    break

            previous_rate = rate
            last_failing_n = n
            next_n = int(n * growth_factor)
            if next_n > max_n:
                hit_max_n = True
                break
            n = next_n

        curve = list(search_path)
        required_n = first_passing_n

        if refine and first_passing_n is not None and last_failing_n is not None:
            lo, hi = last_failing_n, first_passing_n
            for _ in range(refine_steps):
                mid = (lo + hi) // 2
                if mid <= lo or mid >= hi:
                    break
                rng = np.random.RandomState(rng_seed); rng_seed += 1
                mid_rate = self._detection_rate_at_n(mid, true_p_before, effect_size, n_comparisons, n_trials, rng)
                curve.append({"sample_size": mid, "detection_rate": mid_rate})
                if mid_rate >= target_power:
                    hi = mid; required_n = mid
                else:
                    lo = mid

        curve.sort(key=lambda row: row["sample_size"])
        return {"true_p_before": true_p_before, "effect_size": effect_size, "target_power": target_power,
                "required_n": required_n, "hit_max_n": hit_max_n, "diminishing_returns": diminishing_returns,
                "search_path": search_path, "curve": curve, "n_trials_per_point": n_trials}


adaptive_optimizer = SampleSizeOptimizerAdaptive(confidence_level=0.95, n_bootstrap_iterations=800)
result_adaptive = adaptive_optimizer.find_required_n_adaptive(
    true_p_before=0.70, effect_size=0.03, target_power=0.80, start_n=500, max_n=50000, n_trials=200,
)
print("Adaptive search for the 3% effect (previously unresolved on a fixed grid up to 3000):")
for row in result_adaptive["curve"]:
    print(f"  n={row['sample_size']:>5}  detection_rate={row['detection_rate']:.0%}")
print(f"\nRequired n: {result_adaptive['required_n']}")
if result_adaptive["required_n"] is not None:
    cost = result_adaptive["required_n"] * 0.02
    print(f"At $0.02/query (example LLM-judge cost), estimated cost: ${cost:,.2f}")


Adaptive search for the 3% effect (previously unresolved on a fixed grid up to 3000):
  n=  500  detection_rate=22%
  n= 1000  detection_rate=39%
  n= 2000  detection_rate=54%
  n= 3000  detection_rate=69%
  n= 3500  detection_rate=76%
  n= 3562  detection_rate=78%
  n= 3593  detection_rate=79%
  n= 3625  detection_rate=84%
  n= 3750  detection_rate=86%
  n= 4000  detection_rate=86%

Required n: 3625
At $0.02/query (example LLM-judge cost), estimated cost: $72.50


**Result:** the adaptive search converges on **~3,600-4,000** as the true
required sample size for a 3% effect at this baseline rate -- roughly
2.4x the original "n=1500+" estimate. Two independent runs (different
`n_trials`) converged on consistent numbers in this range, giving real
confidence in the corrected figure. This is the concrete, quantified
correction to the unverified planning table this section started from.


---
## Summary of Results

| Version | What it added | Key finding |
|---|---|---|
| v1 | Raw paired bootstrap script | A real +5% effect can come back non-significant at n=200 -- sampling noise, not a bug |
| v2 | `AIBootstrapAuditor` class | Reusable across binary and continuous metrics; vectorized (~10-50x faster) |
| v3 | `AuditBatch` + Bonferroni | Automatic multiple-comparisons correction, no longer relies on callers remembering `n_comparisons` |
| v4 | Benjamini-Hochberg option | Less conservative for larger batches; explicit disclosure that CIs stay uncorrected in this mode |
| v5 | Core-vs-guardrail routing critique | Bonferroni-for-guardrails is backwards -- raises the bar exactly where sensitivity is needed most |
| v6 | `GuardrailAuditor` | Non-inferiority testing with a genuine PASS/FAIL/**INCONCLUSIVE** third state |
| v7 | Unified `AuditBatch` + guardrails | One report, two statistically distinct treatments, matching the corrected diagram |
| v8 | `HarnessCalibrationSuite` | Caught and fixed a real substring bug (`"GO" in "NO-GO"`) that produced a meaningless 100% false-positive rate |
| v9 | `SampleSizeOptimizer` (fixed grid) | Confirmed a 15% effect estimate; disproved a 2-3% effect estimate (required n was higher than claimed) |
| v10 | Adaptive search + diminishing-returns short-circuit | Converged on ~3,600-4,000 as the corrected required n for the 3% case, ~2.4x the original unverified estimate |

## Limitations and Future Work

- **Binary/proportion metrics dominate the demos.** Continuous metrics
  (latency) are supported identically (mean-based bootstrap works for
  either), but the calibration and sample-size search sections default to
  `np.random.binomial` scenarios; a continuous-metric calibration pass
  would strengthen the evidence base further.
- **BH-mode confidence intervals are uncorrected**, disclosed explicitly in
  the risk statement -- a documented scope limit, not a hidden gap.
- **Guardrail alpha is not corrected across a growing batch of guardrails
  by default** -- deliberate, since guardrails should stay sensitive as
  more are added, not stricter. If batches of guardrails grow very large,
  a lighter-touch correction specifically on the FAIL side (not the PASS
  side) would be the next thing to consider, not blanket Bonferroni.
- **Sample-size search precision scales with `n_trials`.** Fast exploratory
  runs (`n_trials=30`) are noisy (see the non-monotonic curves in the
  notebook's own outputs); final planning numbers should always be
  re-verified at `n_trials=200+` before being used to justify real budget,
  exactly as demonstrated in v9-v10 above.
- **Interactive deployment** (Streamlit app with a fast-path/slow-path
  split, `st.cache_data` wiring, and a `to_dataframe()` refactor for every
  class to avoid stdout-parsing) was built and tested separately from this
  notebook; see the accompanying `audit_toolkit_app.py`,
  `slow_path_cache.py`, and `ai_bootstrap_auditor.py` files. A real caching
  bug was caught there too (Streamlit's `st.cache_data` silently excludes
  any parameter with a leading underscore from its cache key, which
  defeated an initial `_cache_bust` cache-invalidation parameter) --
  documented in that codebase's history for the same reason the bugs above
  are kept here: catching and disclosing a real mistake is more useful to
  a reader than presenting only a version that already works.
